# Part I Capstone — PCA from Scratch

Companion notebook for the Part I capstone of *The Math That Powers AI* (2nd ed) — all functions are imported from the repo's `mathpowersai` package (`src/mathpowersai/pca.py`).

The default path is **offline and deterministic**: it runs the full pipeline on a seeded synthetic low-rank-plus-noise dataset (`np.random.default_rng(42)`), exactly like the example script's default mode (`examples/capstone_pca.py`). The book's MNIST run and the scikit-learn cross-check live in an optional cell at the very end; the notebook executes fully without scikit-learn or an internet connection.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

from mathpowersai.pca import (
    PCA,
    pca_via_eigendecomposition,
    pca_via_svd,
    compare_methods,
    variance_threshold_components,
)

## Synthetic data: seeded low-rank plus noise

We build $X = ZW + \text{noise} + \text{offset}$, where $Z$ is $(n_{\text{samples}}, r)$ and $W$ is $(r, n_{\text{features}})$ with $r = 5$, so the signal lives in a rank-5 subspace of the 30-dimensional feature space, plus small isotropic Gaussian noise. With `np.random.default_rng(42)` the run is fully deterministic — the chapter's claims about variance concentration are checkable to the digit.

In [ ]:
rng = np.random.default_rng(42)

n_samples, n_features, rank, noise_scale = 500, 30, 5, 0.1
Z = rng.normal(size=(n_samples, rank))
W = rng.normal(size=(rank, n_features))
noise = noise_scale * rng.normal(size=(n_samples, n_features))
offset = rng.normal(size=n_features)
X = Z @ W + noise + offset

print(f"Data shape: {X.shape}  (true signal rank: {rank})")

## Fit PCA on the synthetic data

`PCA.fit` centers the data, takes the SVD of the centered matrix (avoiding $X^\top X$, which squares the condition number), and stores the principal axes as rows of `components_`. The explained variances are the covariance eigenvalues recovered from singular values with Bessel's correction, $\lambda_i = \sigma_i^2 / (n - 1)$. Since the signal is rank 5, five components should capture nearly all the variance.

In [ ]:
pca = PCA(n_components=5)
X_proj = pca.fit_transform(X)

print(f"Projected shape: {X.shape} -> {X_proj.shape}")
print(f"components_ shape: {pca.components_.shape}")
print("Explained variance (lambda_i = sigma_i^2 / (n-1)):")
print(" ", np.round(pca.explained_variance_, 4))
print("Explained variance ratio:")
print(" ", np.round(pca.explained_variance_ratio_, 4))
print(f"Variance captured by {pca.n_components_} PCs: "
      f"{pca.explained_variance_ratio_.sum() * 100:.2f}%")

## Eigendecomposition vs SVD

The chapter derives PCA two ways: eigendecomposition of the sample covariance $\Sigma = \frac{1}{n-1} X^\top X$ (via `np.linalg.eigh`), and SVD of the centered data matrix with $\lambda_i = \sigma_i^2 / (n - 1)$. `compare_methods` checks that the two agree — eigenvalues to high precision, and eigenvectors up to sign (the sign of an eigenvector is arbitrary).

In [ ]:
X_centered = X - np.mean(X, axis=0)

vals_eig, vecs_eig = pca_via_eigendecomposition(X_centered)
vals_svd, vecs_svd = pca_via_svd(X_centered)

result = compare_methods(vals_eig, vecs_eig, vals_svd, vecs_svd)

## Explained-variance threshold sweep

The chapter's output box asks: how many components are needed to explain at least 50%, 80%, 90%, 95%, and 99% of the total variance? `variance_threshold_components` fits a full PCA and, for each threshold $t$, returns the smallest $k$ with $\sum_{i \le k} \lambda_i / \sum_i \lambda_i \ge t$. On rank-5-plus-noise data, every threshold up to 99% should be met within the first five components.

In [ ]:
thresholds = [0.5, 0.8, 0.9, 0.95, 0.99]
sweep = variance_threshold_components(X, thresholds)

print("Components needed for variance thresholds:")
for t in thresholds:
    print(f"  {t * 100:5.1f}%: {sweep[t]:3d} components")

## Reconstruction error vs number of components

Projecting onto $k$ components and mapping back (`inverse_transform`) gives the best rank-$k$ approximation of the centered data (Eckart–Young). The mean squared reconstruction error therefore equals the variance left out, $\sum_{i > k} \lambda_i / d$ — it should drop sharply until $k$ reaches the signal rank (5), then decay only slowly as we start fitting noise, hitting zero at $k = 30$.

In [ ]:
print("Reconstruction MSE vs number of components:")
for k in [1, 2, 3, 4, 5, 10, 20, 30]:
    pca_k = PCA(n_components=k)
    X_recon = pca_k.inverse_transform(pca_k.fit_transform(X))
    mse = float(np.mean((X - X_recon) ** 2))
    captured = float(np.sum(pca_k.explained_variance_ratio_))
    print(f"  k={k:3d}: MSE = {mse:.6f}   "
          f"(variance explained: {captured * 100:6.2f}%)")

## Optional: scikit-learn validation and the book's MNIST run

**This cell is optional and is the only part of the notebook that touches the network.** If scikit-learn is installed, it first cross-checks the from-scratch PCA against `sklearn.decomposition.PCA` on the synthetic data (`np.allclose` on `explained_variance_ratio_`), then attempts the book's full MNIST demo via `fetch_openml`, which **downloads ~55 MB** on first use. If scikit-learn is missing, or the download fails (e.g. no internet), the cell prints a skip message instead of raising — the rest of the notebook is unaffected.

In [ ]:
# OPTIONAL CELL -- requires scikit-learn; the MNIST fetch downloads data.
try:
    from sklearn.datasets import fetch_openml
    from sklearn.decomposition import PCA as SklearnPCA
except ImportError:
    print("scikit-learn not installed; skipping the optional "
          "sklearn validation and MNIST demo.")
else:
    # 1) Validate the from-scratch PCA against sklearn on the
    #    synthetic data (no network needed for this part).
    ours = PCA(n_components=5).fit(X)
    theirs = SklearnPCA(n_components=5).fit(X)
    ratio_match = np.allclose(
        ours.explained_variance_ratio_,
        theirs.explained_variance_ratio_,
        rtol=1e-6, atol=1e-10,
    )
    print(f"explained_variance_ratio_ matches sklearn: {ratio_match}")

    # 2) The book's MNIST run (downloads ~55MB on first use).
    try:
        print("Loading MNIST dataset (may download ~55MB)...")
        mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    except Exception as exc:
        print(f"Skipping the MNIST demo (fetch failed: {exc!r}). "
              "This is expected without internet access.")
    else:
        X_full = mnist.data.astype(np.float64)
        # Use a 10,000-image subset for speed, as in the book.
        idx = np.random.default_rng(42).choice(
            X_full.shape[0], 10000, replace=False
        )
        X_mnist = X_full[idx]
        print(f"MNIST subset shape: {X_mnist.shape}")

        pca_mnist = PCA(n_components=50)
        X_proj_m = pca_mnist.fit_transform(X_mnist)
        X_recon_m = pca_mnist.inverse_transform(X_proj_m)
        mse_m = float(np.mean((X_mnist - X_recon_m) ** 2))
        captured_m = float(np.sum(pca_mnist.explained_variance_ratio_))
        print(f"Reduced 784 -> 50 dims; variance explained: "
              f"{captured_m * 100:.1f}%")
        print(f"Reconstruction MSE (50 PCs): {mse_m:.2f}")

        sweep_m = variance_threshold_components(X_mnist, thresholds)
        print("Components needed for variance thresholds (MNIST):")
        for t in thresholds:
            print(f"  {t * 100:5.1f}%: {sweep_m[t]:3d} components")

        sk_mnist = SklearnPCA(n_components=50).fit(X_mnist)
        mnist_match = np.allclose(
            pca_mnist.explained_variance_ratio_,
            sk_mnist.explained_variance_ratio_,
            rtol=1e-6, atol=1e-10,
        )
        print(f"MNIST explained_variance_ratio_ matches sklearn: "
              f"{mnist_match}")